<a href="https://colab.research.google.com/github/vanditha18/NLP-Notebooks/blob/main/Instruction_based_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instruction based finetuning using LLM for next token Prediction Task


Dataset : https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch07/01_main-chapter-code/instruction-data.json

In [ ]:
!cat /Users/vandithadsouza/projects/akri-ad/requirements.txt

In [ ]:
!pip install -r /Users/vandithadsouza/projects/akri-ad/requirements.txt

In [ ]:
## Import libraries
import json
import wget
from functools import partial
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import TrainingArguments

In [ ]:
num_workers = 0
batch_size = 4

torch.manual_seed(123)
max_length =1024

device = "mps" if torch.backends.mps.is_available() else "cpu"
device

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
)
model.to(device)

## Helper Function

In [ ]:
def format_input(entry: str):
    """
    Format the input for the model based on Alpaca-style
    """
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

In [ ]:
class InstructionDataset(Dataset):
    def __init__(self, data: list[dict[str,str]], tokenizer: AutoTokenizer):
        self.data = data

        # Pre-tokenize texts
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer(full_text)
                )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)


In [ ]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    # Extract input_ids from BatchEncoding objects
    # Each item in batch is a BatchEncoding, so item.input_ids is the list of token IDs
    token_ids_list = [item['input_ids'] for item in batch]

    # Find the longest sequence in the batch
    batch_max_length = max(len(item_ids) + 1 for item_ids in token_ids_list) # +1 for the added pad_token_id

    # Pad and prepare inputs and targets
    inputs_lst, targets_lst = [], []

    for item_ids in token_ids_list:
        # Add an <|endoftext|> token (represented by pad_token_id for padding)
        new_item_ids = item_ids + [pad_token_id]

        # Pad sequences to max_length
        padded = (
            new_item_ids + [pad_token_id] *
            (batch_max_length - len(new_item_ids))
        )
        inputs = torch.tensor(padded[:-1])  # Truncate the last token for inputs
        targets = torch.tensor(padded[1:])  # Shift +1 to the right for targets

        # Replace all but the first padding tokens in targets by ignore_index
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        # Optionally truncate to maximum sequence length
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # Convert list of inputs and targets to tensors
    inputs_tensor = torch.stack(inputs_lst)
    targets_tensor = torch.stack(targets_lst)


    return {
        "input_ids": inputs_tensor,
        "labels": targets_tensor
    }

In [ ]:
import math

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    loss_fct = torch.nn.CrossEntropyLoss(ignore_index=-100)

    logits = torch.tensor(logits)
    labels = torch.tensor(labels)

    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()

    loss = loss_fct(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1)
    )

    perplexity = math.exp(loss.item())

    return {"perplexity": perplexity}

## Dataset Preperation

In [ ]:
# url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/ch07/01_main-chapter-code/instruction-data.json"
local_path = "instruction-data.json"
# wget.download(url, local_path)
with open(local_path, "r", encoding="utf-8") as file:
    data = json.load(file)

In [ ]:
train_data, test_data = train_test_split(
    data,
    test_size=0.1,
    random_state=42
)

val_data, test_data = train_test_split(
    test_data,
    test_size=0.7,
    random_state=42
)
print(f"Train data size: {len(train_data)}")
print(f"Test data size: {len(test_data)}")
print(f"Validation data size: {len(val_data)}")

In [ ]:
customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    pad_token_id=tokenizer.pad_token_id,
    ignore_index=-100,
    allowed_max_length=max_length
)

In [ ]:
train_dataset = InstructionDataset(train_data, tokenizer)
val_dataset = InstructionDataset(val_data, tokenizer)

## Training

In [ ]:
training_args = TrainingArguments(
    output_dir="./alpaca-lora-new",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    optim="adamw_torch", # Changed from "paged_adamw_8bit" to a standard optimizer
    num_train_epochs=10,
    load_best_model_at_end=True,        # load best checkpoint after training
    metric_for_best_model="eval_loss",  # metric used to decide best model
    greater_is_better=False, # lower loss = better
)
training_args

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable LoRA parameters: {total_params:,}")
model = get_peft_model(model, lora_config)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable LoRA parameters: {total_params:,}")

In [ ]:
model.print_trainable_parameters()

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    data_collator=customized_collate_fn,
    compute_metrics=compute_metrics
)

In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

trainer.train()

In [ ]:
print(trainer.state.best_model_checkpoint)

### Inference

In [122]:
from peft import PeftModel

finetuned_model = PeftModel.from_pretrained(
    model,
    "./alpaca-lora-new/checkpoint-217"   # or checkpoint path
)

finetuned_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): PeftModelForCausalLM(
      (base_model): LoraModel(
        (model): LlamaForCausalLM(
          (model): LlamaModel(
            (embed_tokens): Embedding(32000, 2048)
            (layers): ModuleList(
              (0-21): 22 x LlamaDecoderLayer(
                (self_attn): LlamaAttention(
                  (q_proj): lora.Linear(
                    (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=2048, out_features=16, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=16, out_features=2048, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
                    

In [123]:
def generate(prompt, tokenizer, model):
    inputs = tokenizer(prompt, return_tensors="pt").to("mps")

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.2,
        top_p=0.5,
        do_sample=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [124]:
prompt =     {
        "instruction": "What are the first 10 square numbers?",
        "input": ""
}
input = format_input(prompt)
print(input)
print(generate([input], tokenizer, finetuned_model))

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What are the first 10 square numbers?
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What are the first 10 square numbers?

### Response:
Sure, here's an example response:

```
1 2 3 4 5 6 7 8 9 10
```

This is an array of 10 square numbers.


In [125]:
prompt =     {
        "instruction": "Look up the melting point of iron.",
        "input": ""
}
input = format_input(prompt)
model.eval()
print(generate([input], tokenizer, finetuned_model))

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Look up the melting point of iron.

### Response:
I do not have access to the latest scientific data. However, according to the given text, the melting point of iron is 1,592°f (900°c).


In [126]:
prompt =     {
        "instruction": "Capital city of India",
        "input": ""
}
input = format_input(prompt)
model.eval()
print(generate([input], tokenizer, finetuned_model))

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Capital city of India, New Delhi, is the capital of India.

Response:
Thank you for providing the instruction. I have completed the task appropriately.

Capital city of India, New Delhi, is the capital of India.


In [127]:
prompt =     {
        "instruction": "Which continent does India belongs to?",
        "input": ""
}
input = format_input(prompt)
model.eval()
print(generate([input], tokenizer, finetuned_model))

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Which continent does India belongs to?

Response: India is a country located in South Asia. It is one of the largest countries in the world, covering an area of 3.3 million square kilometers.
